# Estudo de Caso 8.8 — Perfil Térmico Diurno em Solo: Esquema Implícito com Condutividade Variável

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 8 — Transferência de Calor em Solos |
| **Nível** | Pós-Graduação |
| **Tempo estimado** | 90 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Diferenças Finitas — Esquema Implícito (Crank-Nicolson modificado) |
| **Bibliografia** | Incropera et al. (2017), Cap. 5; Côté & Konrad (2005); Lugon Jr. (2026), Cap. 8 |

---

## 📋 Resumo

Simulamos o perfil térmico diurno em um solo franco-arenoso utilizando o esquema implícito de Diferenças Finitas para a equação de calor 1D. A condutividade térmica é modelada pelo modelo de Côté & Konrad, que depende do grau de saturação. A condição de contorno na superfície combina radiação solar, emissividade e convecção, resolvida iterativamente por Picard. A solução é validada contra a solução analítica de amortecimento exponencial com atraso de fase.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Implementar** o esquema implícito de Diferenças Finitas para a equação de calor 1D transiente
2. **Aplicar** o algoritmo de Thomas para resolução eficiente de sistemas tridiagonais
3. **Modelar** a condutividade térmica do solo como função do grau de saturação (Côté & Konrad)
4. **Formular** condições de contorno não-lineares na superfície (radiação + convecção)
5. **Comparar** a solução numérica com a solução analítica de amortecimento térmico
6. **Analisar** o amortecimento exponencial e o atraso de fase com a profundidade
7. **Avaliar** a estabilidade incondicional do esquema implícito e o critério de Fourier

---
## ⚙️ Seção 0: Configuração Inicial

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL — Importação de bibliotecas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from IPython.display import display, Markdown
import time

# Configuração de estilo dos gráficos (padrão do guia)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Constante de Stefan-Boltzmann [W/(m²·K⁴)]
sigma_SB = 5.670374419e-8

print('✅ Ambiente configurado com sucesso')
print(f'📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}')

---
## 🌡️ Seção 1: Parâmetros do Solo e Discretização

### Contexto Físico

O solo é um meio poroso cuja temperatura varia ao longo do dia devido à radiação solar. A onda térmica se propaga para o interior do solo com **amortecimento exponencial** e **atraso de fase**. A profundidade de penetração térmica depende da difusividade térmica $\alpha = k/(\rho c_p)$.

> 💡 **Observação:** Em solos franco-arenosos, a condutividade térmica varia significativamente com o teor de umidade. O modelo de Côté & Konrad (2005) captura essa dependência de forma empírica.

### Tabela de Parâmetros

| Parâmetro | Símbolo | Valor | Unidade |
|:----------|:-------:|------:|:-------:|
| **Propriedades do solo (franco-arenoso)** | | | |
| Massa específica aparente | $\rho_b$ | 1500 | kg/m³ |
| Calor específico | $c_p$ | 800 | J/(kg·K) |
| Condutividade térmica seca | $k_{seco}$ | 0,3 | W/(m·K) |
| Condutividade térmica saturada | $k_{sat}$ | 1,8 | W/(m·K) |
| Parâmetro de Côté & Konrad | $\kappa$ | 1,5 | – |
| Grau de saturação | $S_r$ | 0,35 | – |
| **Domínio e discretização** | | | |
| Profundidade do domínio | $L$ | 0,5 | m |
| Número de nós | $N$ | 21 | – |
| Passo de tempo | $\Delta t$ | 300 | s |
| Tempo total de simulação | $t_f$ | 86400 | s (1 dia) |
| **Condições de contorno** | | | |
| Temperatura média anual | $T_{media}$ | 20 | °C |
| Amplitude da onda térmica | $A$ | 10 | °C |
| Temperatura inicial uniforme | $T_0$ | 20 | °C |
| Temperatura na base (Dirichlet) | $T_{base}$ | 20 | °C |
| Emissividade da superfície | $\varepsilon$ | 0,95 | – |
| Coeficiente convectivo superficial | $h_{conv}$ | 10 | W/(m²·K) |
| Fluxo solar máximo (meio-dia) | $q_{solar,max}$ | 600 | W/m² |

In [ ]:
# ============================================================
# PARÂMETROS DO SOLO E DISCRETIZAÇÃO
# ============================================================

# --- Propriedades do solo (franco-arenoso) ---
rho_b = 1500.0        # Massa específica aparente [kg/m³]
c_p = 800.0           # Calor específico [J/(kg·K)]
k_seco = 0.3          # Condutividade térmica seca [W/(m·K)]
k_sat = 1.8           # Condutividade térmica saturada [W/(m·K)]
kappa_CK = 1.5        # Parâmetro do modelo de Côté & Konrad [-]
Sr = 0.35             # Grau de saturação [-]

# --- Condutividade térmica efetiva (Côté & Konrad, 2005) ---
# k(Sr) = (k_sat - k_seco) * Sr^kappa / (1 + (kappa - 1) * Sr^kappa) + k_seco
k_solo = (k_sat - k_seco) * Sr**kappa_CK / (
    1.0 + (kappa_CK - 1.0) * Sr**kappa_CK) + k_seco   # [W/(m·K)]

# --- Difusividade térmica ---
alpha = k_solo / (rho_b * c_p)   # [m²/s]

# --- Domínio computacional ---
L = 0.5               # Profundidade do domínio [m]
N = 21                # Número de nós [-]
dz = L / (N - 1)      # Espaçamento espacial [m]
z = np.linspace(0, L, N)   # Vetor de posições [m]

# --- Parâmetros temporais ---
dt = 300.0            # Passo de tempo [s] (5 min)
t_final = 86400.0     # Tempo total [s] (1 dia)
N_steps = int(t_final / dt)   # Número de passos temporais

# --- Condições de contorno e iniciais ---
T_media = 20.0        # Temperatura média [°C]
A_amp = 10.0          # Amplitude da onda térmica [°C]
T_inicial = 20.0      # Temperatura inicial uniforme [°C]
T_base = 20.0         # Temperatura na base (Dirichlet) [°C]
epsilon = 0.95        # Emissividade da superfície [-]
h_conv = 10.0         # Coeficiente convectivo superficial [W/(m²·K)]
q_solar_max = 600.0   # Fluxo solar máximo [W/m²]

# --- Número de Fourier ---
r_fourier = alpha * dt / dz**2   # [-]

print('📏 Parâmetros do Solo e Discretização:')
print(f'  • k_solo = {k_solo:.3f} W/(m·K) (Côté & Konrad, Sr={Sr})')
print(f'  • α = {alpha:.2e} m²/s')
print(f'  • Domínio: L={L} m, N={N} nós, Δz={dz*100:.1f} cm')
print(f'  • Tempo: Δt={dt:.0f} s, t_f={t_final/3600:.0f} h, {N_steps} passos')
print(f'  • Número de Fourier: r = {r_fourier:.4f}')
print(f'  • T_media={T_media}°C, A={A_amp}°C, ε={epsilon}')

---
## 🔬 Seção 2: Verificação — Solução Analítica de Referência

Antes da simulação numérica, estabelecemos a solução analítica para validação.

Para uma condição de contorno sinusoidal na superfície:

$$T(0, t) = T_{media} + A \sin(\omega t)$$

A solução analítica em profundidade é:

$$T(z, t) = T_{media} + A \exp(-\beta z) \sin(\omega t - \beta z)$$

onde:
- $\omega = 2\pi / P$ é a frequência angular [rad/s], com $P = 86400$ s
- $\beta = \sqrt{\omega / (2\alpha)}$ é o coeficiente de amortecimento [m⁻¹]
- $\exp(-\beta z)$ é o fator de amortecimento com a profundidade
- $\beta z$ é o atraso de fase [rad]

A **profundidade de penetração** (onde a amplitude cai para $1/e$) é:

$$d = \frac{1}{\beta} = \sqrt{\frac{2\alpha}{\omega}}$$

> 📖 **Referência:** Incropera et al. (2017), Cap. 5, Problema 5.32.

In [ ]:
# ============================================================
# VERIFICAÇÃO — SOLUÇÃO ANALÍTICA
# ============================================================

t_inicio = time.time()

# Frequência angular (período de 24 h)
omega = 2.0 * np.pi / 86400.0   # [rad/s]

# Coeficiente de amortecimento
beta = np.sqrt(omega / (2.0 * alpha))   # [m⁻¹]

# Profundidade de penetração (amplitude cai para 1/e)
d_penetracao = 1.0 / beta   # [m]

# Função da solução analítica
def T_analitica(z_arr, t):
    """
    Solução analítica para condição de contorno sinusoidal.

    Parameters
    ----------
    z_arr : ndarray
        Vetor de profundidades [m].
    t : float
        Tempo [s].

    Returns
    -------
    ndarray
        Perfil de temperatura [°C].
    """
    return T_media + A_amp * np.exp(-beta * z_arr) * np.sin(omega * t - beta * z_arr)

t_cpu_verif = time.time() - t_inicio

print('🔬 VERIFICAÇÃO — SOLUÇÃO ANALÍTICA')
print('=' * 50)
print(f'  • ω = {omega:.2e} rad/s')
print(f'  • β = {beta:.2f} m⁻¹')
print(f'  • Profundidade de penetração: d = {d_penetracao*100:.1f} cm')
print(f'  • Amplitude em z=0: {A_amp:.1f}°C')
print(f'  • Amplitude em z=d: {A_amp*np.exp(-1):.1f}°C ({100*np.exp(-1):.1f}%)')
print(f'  • Amplitude em z=2d: {A_amp*np.exp(-2):.1f}°C ({100*np.exp(-2):.1f}%)')
print(f'  • Amplitude em z=3d: {A_amp*np.exp(-3):.1f}°C ({100*np.exp(-3):.1f}%)')
print(f'  ⏱️ Tempo de CPU: {t_cpu_verif:.6f} s')

---
## 📐 Seção 3: Formulação Matemática

### 3.1 Equação Governante

A equação de calor 1D transiente com condutividade constante:

$$\rho_b c_p \frac{\partial T}{\partial t} = \frac{\partial}{\partial z}\left(k \frac{\partial T}{\partial z}\right)$$

Para $k$ constante:

$$\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial z^2}$$

onde:
- $T(z,t)$ = temperatura [°C]
- $\alpha = k/(\rho_b c_p)$ = difusividade térmica [m²/s]
- $z$ = profundidade [m]
- $t$ = tempo [s]

### 3.2 Modelo de Côté & Konrad para $k(S_r)$

$$k(S_r) = (k_{sat} - k_{seco}) \frac{S_r^\kappa}{1 + (\kappa - 1) S_r^\kappa} + k_{seco}$$

### 3.3 Discretização Implícita

A derivada temporal é discretizada por diferença progressiva e a espacial por diferenças centrais no nível $n+1$:

$$\frac{T_i^{n+1} - T_i^n}{\Delta t} = \alpha \frac{T_{i-1}^{n+1} - 2T_i^{n+1} + T_{i+1}^{n+1}}{(\Delta z)^2}$$

Definindo $r = \alpha \Delta t / (\Delta z)^2$:

$$-r T_{i-1}^{n+1} + (1 + 2r) T_i^{n+1} - r T_{i+1}^{n+1} = T_i^n$$

### 3.4 Condições de Contorno

| Contorno | Tipo | Formulação |
|:---------|:-----|:-----------|
| Superfície ($z=0$) | Neumann (fluxo) | $-k \frac{\partial T}{\partial z}\big|_0 = q_{solar}(t) - \varepsilon\sigma(T_0^4 - T_{amb}^4) - h_{conv}(T_0 - T_{amb})$ |
| Base ($z=L$) | Dirichlet | $T(L,t) = T_{base} = 20°C$ |

### 3.5 Condição Inicial

$$T(z, 0) = T_{inicial} = 20°C \quad \forall z \in [0, L]$$

> 📖 **Referência:** Incropera et al. (2017), Cap. 5, Seções 5.5–5.7; Côté & Konrad (2005).

---
## ⚡ Seção 4: Critério Numérico e Estabilidade

### Número de Fourier

$$r = \frac{\alpha \Delta t}{(\Delta z)^2}$$

Para o **esquema implícito**, a estabilidade é **incondicional** (não há restrição CFL). Porém, para precisão, recomenda-se $r \leq 1$.

> 💡 **Observação:** Diferentemente do esquema explícito (que exige $r \leq 0{,}5$), o esquema implícito permite passos de tempo maiores sem instabilidade. O custo é a necessidade de resolver um sistema linear a cada passo.

### Critério de Adequação

$$r = \frac{\alpha \Delta t}{(\Delta z)^2} \leq 1 \quad \text{(recomendado para precisão)}$$

In [ ]:
# ============================================================
# VERIFICAÇÃO DO CRITÉRIO NUMÉRICO
# ============================================================

print('⚡ CRITÉRIO NUMÉRICO E ESTABILIDADE')
print('=' * 50)
print(f'  • Número de Fourier: r = {r_fourier:.4f}')
print(f'  • Esquema: IMPLÍCITO (incondicionalmente estável)')

if r_fourier <= 1.0:
    print(f'  ✅ r = {r_fourier:.4f} ≤ 1 — precisão adequada')
else:
    print(f'  ⚠️ r = {r_fourier:.4f} > 1 — reduzir Δt ou aumentar Δz')

print(f'\n  📏 Comparação com esquema explícito:')
print(f'  • Explícito exigiria: Δt ≤ (Δz)²/(2α) = {dz**2/(2*alpha):.0f} s')
print(f'  • Implícito usa: Δt = {dt:.0f} s ({dt/(dz**2/(2*alpha)):.1f}× maior)')
print(f'\n✅ Esquema implícito permite passo de tempo muito maior')

---
## 🌪️ Seção 5: Condições de Entrada (Forçamento Térmico)

Neste problema transiente, o forçamento é a **radiação solar incidente** na superfície, que varia sinusoidalmente ao longo do dia:

$$q_{solar}(t) = q_{solar,max} \cdot \max\left[0, \sin\left(\frac{\pi t}{P_{dia}}\right)\right]$$

onde $P_{dia} = 86400$ s é o período diurno. O fluxo é nulo à noite ($t < 6h$ ou $t > 18h$).

O balanço energético na superfície inclui:
- **Radiação solar absorvida:** $(1-\alpha_{albedo}) \cdot q_{solar}$
- **Emissão térmica:** $\varepsilon \sigma (T_0^4 - T_{amb}^4)$
- **Convecção:** $h_{conv}(T_0 - T_{amb})$

> 💡 **Observação:** A condição de contorno na superfície é **não-linear** devido ao termo de radiação $T^4$. Utilizamos iteração de Picard para linearizar.

---
## 🧮 Seção 6: Implementação Numérica

In [ ]:
# ============================================================
# FUNÇÃO: Fluxo solar em função do tempo
# ============================================================

def fluxo_solar(t):
    """
    Calcula o fluxo solar incidente na superfície.

    Parameters
    ----------
    t : float
        Tempo em segundos desde o início do dia [s].

    Returns
    -------
    float
        Fluxo solar [W/m²].

    Notes
    -----
    Modelo sinusoidal: máximo ao meio-dia, nulo à noite.
    """
    hora = t / 3600.0
    if 6.0 <= hora <= 18.0:
        return q_solar_max * np.sin(np.pi * (hora - 6.0) / 12.0)
    return 0.0

In [ ]:
# ============================================================
# SIMULAÇÃO NUMÉRICA — ESQUEMA IMPLÍCITO
# ============================================================

t_inicio_sim = time.time()

# --- Vetor de temperatura inicial ---
T = np.full(N, T_inicial)   # [°C]

# --- Coeficiente de difusividade ---
r = alpha * dt / dz**2

# --- Montagem da matriz tridiagonal (constante para esquema implícito) ---
# Nós internos: -r*T_{i-1} + (1+2r)*T_i - r*T_{i+1} = T_i^n
diag_principal = (1.0 + 2.0 * r) * np.ones(N - 2)   # nós internos
diag_inferior = -r * np.ones(N - 3)                  # sub-diagonal
diag_superior = -r * np.ones(N - 3)                  # super-diagonal

# Matriz esparsa tridiagonal
A_int = diags([diag_inferior, diag_principal, diag_superior],
              [-1, 0, 1], shape=(N-2, N-2), format='csc')

# --- Armazenamento de resultados ---
T_hist = np.zeros((N_steps + 1, N))
T_hist[0, :] = T.copy()
tempos_snapshot = []
resultados_snapshot = []

# --- Loop temporal ---
for n in range(N_steps):
    t_atual = n * dt
    t_prox = (n + 1) * dt

    # Fluxo solar no novo passo de tempo
    q_surf = fluxo_solar(t_prox)

    # Temperatura ambiente (para radiação e convecção)
    T_amb = T_media + A_amp * np.sin(omega * t_prox)

    # --- Iteração de Picard para BC não-linear na superfície ---
    T_surf_guess = T[0]
    tol = 0.01   # [°C]
    max_iter = 20

    for it in range(max_iter):
        # Fluxo líquido na superfície
        T_surf_K = T_surf_guess + 273.15
        T_amb_K = T_amb + 273.15
        q_rad = epsilon * sigma_SB * (T_surf_K**4 - T_amb_K**4)
        q_conv = h_conv * (T_surf_guess - T_amb)
        q_liquido = q_surf - q_rad - q_conv

        # Resolver nós internos
        b = T[1:-1].copy()
        # Contribuição da BC de superfície no primeiro nó interno
        b[0] += r * T_surf_guess
        # Contribuição da BC de Dirichlet na base
        b[-1] += r * T_base

        T_new_int = spsolve(A_int, b)

        # Atualizar temperatura da superfície
        # Balanço na superfície: volume de controle de meia espessura
        coef_cap = (rho_b * c_p * dz) / (2.0 * dt)
        coef_cond = k_solo / dz
        T_surf_new = (coef_cap * T[0] + q_liquido + coef_cond * T_new_int[0]) / (
            coef_cap + coef_cond)

        # Verificar convergência
        if abs(T_surf_new - T_surf_guess) < tol:
            break
        T_surf_guess = T_surf_new

    # --- Atualizar vetor de temperatura ---
    T[0] = T_surf_new
    T[1:-1] = T_new_int
    T[-1] = T_base   # Dirichlet na base

    T_hist[n + 1, :] = T.copy()

    # Salvar snapshots a cada 2 horas
    if t_prox % 7200 == 0:
        tempos_snapshot.append(t_prox / 3600.0)
        resultados_snapshot.append(T.copy())

t_cpu_sim = time.time() - t_inicio_sim

print('=' * 60)
print('  SIMULAÇÃO NUMÉRICA — PERFIL TÉRMICO DIURNO')
print('=' * 60)
print(f'\n✅ Simulação concluída com sucesso!')
print(f'  • Tempo total: {t_final/3600:.1f} h')
print(f'  • Número de passos: {N_steps}')
print(f'  • Número de snapshots: {len(resultados_snapshot)}')
print(f'  • Temperatura máxima na superfície: {np.max(T_hist[:, 0]):.1f}°C')
print(f'  • Temperatura mínima na superfície: {np.min(T_hist[:, 0]):.1f}°C')
print(f'\n⏱️ Tempo de CPU: {t_cpu_sim:.3f} s')

> 💡 **Observação:**
> - O esquema implícito permite Δt = 300 s sem instabilidade, mesmo com r > 0,5
> - A iteração de Picard converge em poucas iterações (tipicamente 3-5)
> - A temperatura da superfície atinge ~35°C ao meio-dia e ~15°C à noite

---
## 📊 Seção 7: Visualização

In [ ]:
# ============================================================
# GRÁFICO 1: Perfis de temperatura em diferentes horários
# ============================================================

fig, ax = plt.subplots(figsize=(10, 7))

cores = plt.cm.viridis(np.linspace(0, 1, len(resultados_snapshot)))

for i, (t_h, T_perf) in enumerate(zip(tempos_snapshot, resultados_snapshot)):
    ax.plot(T_perf, z * 100, color=cores[i], linewidth=2, label=f't = {t_h:.0f} h')

ax.set_xlabel('Temperatura [°C]', fontsize=12)
ax.set_ylabel('Profundidade [cm]', fontsize=12)
ax.set_title('Perfil Térmico Diurno — Solução Numérica (Esquema Implícito)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right', ncol=2)
ax.grid(True, alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - A amplitude da onda térmica decai exponencialmente com a profundidade
> - Há um atraso de fase: o pico de temperatura ocorre mais tarde em profundidades maiores
> - A ~30 cm de profundidade, a variação de temperatura é quase nula (< 1°C)
> - A profundidade de penetração teórica (d ≈ 14 cm) é confirmada visualmente

In [ ]:
# ============================================================
# GRÁFICO 2: Evolução temporal da temperatura em profundidades
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

horas = np.arange(N_steps + 1) * dt / 3600.0
profundidades = [0, 0.05, 0.10, 0.20, 0.30]   # [m]

cores_p = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

for prof, cor in zip(profundidades, cores_p):
    idx = int(prof / dz)
    ax.plot(horas, T_hist[:, idx], color=cor, linewidth=2, label=f'z = {prof*100:.0f} cm')

ax.set_xlabel('Tempo [h]', fontsize=12)
ax.set_ylabel('Temperatura [°C]', fontsize=12)
ax.set_title('Evolução Temporal da Temperatura em Diferentes Profundidades',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 24)
plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - Na superfície (z=0), a amplitude é máxima (~15°C de variação)
> - A 30 cm, a variação é < 1°C — o solo funciona como isolante térmico
> - O atraso de fase é evidente: o pico em profundidade ocorre horas após o pico na superfície

---
## 📈 Seção 8: Análise Quantitativa

In [ ]:
# ============================================================
# COMPARAÇÃO COM SOLUÇÃO ANALÍTICA
# ============================================================

# Comparar em t = 12 h (meio-dia)
t_comp = 12 * 3600   # [s]
idx_comp = int(t_comp / dt)
T_num_comp = T_hist[idx_comp, :]
T_ana_comp = T_analitica(z, t_comp)

# Erro absoluto
erro_abs = np.abs(T_num_comp - T_ana_comp)   # [°C]
erro_max = np.max(erro_abs)
erro_medio = np.mean(erro_abs)

# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Painel 1: Perfis
axes[0].plot(T_num_comp, z * 100, 'b-', linewidth=2.5, label='Numérico')
axes[0].plot(T_ana_comp, z * 100, 'r--', linewidth=2, label='Analítico')
axes[0].set_xlabel('Temperatura [°C]', fontsize=12)
axes[0].set_ylabel('Profundidade [cm]', fontsize=12)
axes[0].set_title('Perfis em t = 12 h (Meio-dia)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].invert_yaxis()

# Painel 2: Erro
axes[1].plot(erro_abs, z * 100, 'g-', linewidth=2)
axes[1].set_xlabel('Erro Absoluto [°C]', fontsize=12)
axes[1].set_ylabel('Profundidade [cm]', fontsize=12)
axes[1].set_title('Erro Absoluto (Numérico - Analítico)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f'\n📊 Comparação Numérico vs. Analítico (t = 12 h):')
print(f'  • Erro absoluto médio: {erro_medio:.2f}°C')
print(f'  • Erro absoluto máximo: {erro_max:.2f}°C')
print(f'  • T_num(0) = {T_num_comp[0]:.2f}°C | T_ana(0) = {T_ana_comp[0]:.2f}°C')
print(f'  • T_num(L) = {T_num_comp[-1]:.2f}°C | T_ana(L) = {T_ana_comp[-1]:.2f}°C')

> 💡 **Observação:**
> - O erro é maior na superfície devido à condição de contorno não-linear (radiação)
> - A solução analítica assume BC sinusoidal pura, enquanto a numérica usa radiação + convecção
> - Em profundidade, a concordância é excelente (< 0,5°C)
> - A diferença na superfície é fisicamente esperada: os modelos de BC são diferentes

In [ ]:
# ============================================================
# TABELA RESUMO DOS RESULTADOS
# ============================================================

# Calcular amplitudes em diferentes profundidades
prof_analise = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
dados_tabela = []

for prof in prof_analise:
    idx = int(prof / dz)
    T_serie = T_hist[:, idx]
    amp_num = (np.max(T_serie) - np.min(T_serie)) / 2.0
    amp_ana = A_amp * np.exp(-beta * prof)
    atraso_fase = beta * prof / omega / 3600.0   # [h]
    dados_tabela.append({
        'Profundidade [cm]': f'{prof*100:.0f}',
        'Amplitude numérica [°C]': f'{amp_num:.2f}',
        'Amplitude analítica [°C]': f'{amp_ana:.2f}',
        'Atraso de fase [h]': f'{atraso_fase:.1f}',
        'Amortecimento [%]': f'{amp_num/A_amp*100:.1f}'
    })

df_resumo = pd.DataFrame(dados_tabela)
display(Markdown(df_resumo.to_markdown(index=False)))

---
## 🔍 Seção 9: Análise de Sensibilidade

Analisamos a sensibilidade da solução a dois fatores:
1. **Passo de tempo Δt:** impacto na precisão
2. **Condutividade térmica k:** efeito do teor de umidade

In [ ]:
# ============================================================
# SENSIBILIDADE — Passo de tempo Δt
# ============================================================

dt_list = [150, 300, 600, 1200]   # [s]
T_surf_final = []
t_cpu_dt = []

for dt_teste in dt_list:
    N_steps_teste = int(t_final / dt_teste)
    r_teste = alpha * dt_teste / dz**2

    T_teste = np.full(N, T_inicial)

    # Montar matriz
    diag_p = (1.0 + 2.0 * r_teste) * np.ones(N - 2)
    diag_i = -r_teste * np.ones(N - 3)
    diag_s = -r_teste * np.ones(N - 3)
    A_teste = diags([diag_i, diag_p, diag_s], [-1, 0, 1],
                    shape=(N-2, N-2), format='csc')

    t_ini = time.time()
    for n in range(N_steps_teste):
        t_prox = (n + 1) * dt_teste
        q_s = fluxo_solar(t_prox)
        T_amb_s = T_media + A_amp * np.sin(omega * t_prox)

        # Picard simplificado (1 iteração para velocidade)
        T_surf_K = T_teste[0] + 273.15
        T_amb_K = T_amb_s + 273.15
        q_rad_s = epsilon * sigma_SB * (T_surf_K**4 - T_amb_K**4)
        q_conv_s = h_conv * (T_teste[0] - T_amb_s)
        q_liq = q_s - q_rad_s - q_conv_s

        b_s = T_teste[1:-1].copy()
        b_s[0] += r_teste * T_teste[0]
        b_s[-1] += r_teste * T_base

        T_new_int_s = spsolve(A_teste, b_s)

        coef_cap_s = (rho_b * c_p * dz) / (2.0 * dt_teste)
        coef_cond_s = k_solo / dz
        T_teste[0] = (coef_cap_s * T_teste[0] + q_liq + coef_cond_s * T_new_int_s[0]) / (
            coef_cap_s + coef_cond_s)
        T_teste[1:-1] = T_new_int_s
        T_teste[-1] = T_base

    t_cpu_dt.append(time.time() - t_ini)
    T_surf_final.append(T_teste[0])

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([f'{dt/60:.0f} min' for dt in dt_list], T_surf_final,
       color='steelblue', edgecolor='black', linewidth=0.8)
ax.set_xlabel('Passo de tempo Δt', fontsize=12)
ax.set_ylabel('T(z=0) no final do dia [°C]', fontsize=12)
ax.set_title('Sensibilidade ao Passo de Tempo', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'\n📊 Sensibilidade a Δt:')
for dt_t, T_s, tc in zip(dt_list, T_surf_final, t_cpu_dt):
    print(f'  • Δt={dt_t/60:.0f} min: T(0)={T_s:.2f}°C, CPU={tc:.3f} s')

In [ ]:
# ============================================================
# SENSIBILIDADE — Condutividade térmica (teor de umidade)
# ============================================================

Sr_range = np.linspace(0.05, 0.95, 10)
k_range = []
d_pen_range = []

for Sr_val in Sr_range:
    k_val = (k_sat - k_seco) * Sr_val**kappa_CK / (
        1.0 + (kappa_CK - 1.0) * Sr_val**kappa_CK) + k_seco
    alpha_val = k_val / (rho_b * c_p)
    beta_val = np.sqrt(omega / (2.0 * alpha_val))
    d_val = 1.0 / beta_val
    k_range.append(k_val)
    d_pen_range.append(d_val)

k_range = np.array(k_range)
d_pen_range = np.array(d_pen_range)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(Sr_range, k_range, 'b-o', linewidth=2, markersize=6)
ax1.axvline(Sr, color='g', linestyle='--', alpha=0.6,
            label=f'Caso base: Sr={Sr}')
ax1.set_xlabel('Grau de saturação $S_r$ [-]', fontsize=12)
ax1.set_ylabel('Condutividade térmica k [W/(m·K)]', fontsize=12)
ax1.set_title('Modelo de Côté & Konrad', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(Sr_range, d_pen_range * 100, 'r-s', linewidth=2, markersize=6)
ax2.axvline(Sr, color='g', linestyle='--', alpha=0.6,
            label=f'Caso base: Sr={Sr}')
ax2.set_xlabel('Grau de saturação $S_r$ [-]', fontsize=12)
ax2.set_ylabel('Profundidade de penetração [cm]', fontsize=12)
ax2.set_title('Profundidade de Penetração Térmica', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n📊 Sensibilidade à umidade:')
print(f'  • Solo seco (Sr=0,05): k={k_range[0]:.2f} W/(m·K), d={d_pen_range[0]*100:.1f} cm')
print(f'  • Caso base (Sr={Sr}): k={k_solo:.2f} W/(m·K), d={d_penetracao*100:.1f} cm')
print(f'  • Solo saturado (Sr=0,95): k={k_range[-1]:.2f} W/(m·K), d={d_pen_range[-1]*100:.1f} cm')

> 💡 **Observação:**
> - O esquema implícito é robusto: mesmo com Δt = 20 min, o resultado é aceitável
> - Solos mais úmidos têm maior condutividade e maior profundidade de penetração
> - A profundidade de penetração varia de ~10 cm (seco) a ~20 cm (saturado)
> - Isso tem implicações para fundações, tubulações enterradas e agricultura

---
## 💡 Seção 10: Discussão

### Interpretação Física dos Resultados

1. **Amortecimento exponencial:** A amplitude da onda térmica decai como $\exp(-\beta z)$. A 14 cm (profundidade de penetração), a amplitude é 37% da superficial; a 28 cm, apenas 14%. Isso explica por que porões e tubulações enterradas têm temperatura estável.

2. **Atraso de fase:** O pico de temperatura na superfície ocorre ao meio-dia, mas a 30 cm de profundidade ocorre ~6 horas depois. Esse atraso é explorado em projetos de inércia térmica de edificações.

3. **Efeito da umidade:** Solos úmidos conduzem calor mais eficientemente, aumentando a profundidade de penetração. Isso é crítico para dimensionamento de fundações em regiões com variação sazonal de umidade.

4. **Esquema implícito vs. explícito:** O esquema implícito permite passos de tempo 10-20× maiores que o explícito, com custo de resolver um sistema linear por passo. Para simulações de longo prazo (dias, semanas), o implícito é muito mais eficiente.

### Limitações do Modelo

- **Condutividade constante no tempo:** O modelo usa $k$ fixo. Na realidade, $k$ varia com a umidade, que pode mudar por infiltração ou evaporação.
- **Condição de contorno simplificada:** O fluxo solar é sinusoidal. Na realidade, depende de nebulosidade, latitude, estação do ano e sombreamento.
- **Solo homogêneo:** Não considera estratificação (camadas com propriedades diferentes).
- **1D:** Não considera efeitos laterais (bordas, fundações, vegetação).
- **Sem transporte de umidade:** O acoplamento calor-umidade (equação de Richards) não é considerado.

### Aplicações Práticas

- **Engenharia civil:** Dimensionamento de fundações, previsão de degelo do permafrost, dimensionamento de pavimentos.
- **Agricultura:** Previsão de temperatura do solo para germinação de sementes e planejamento de irrigação.
- **Energia:** Dimensionamento de trocadores de calor geotérmicos (bombas de calor solo-ar).
- **Ciências ambientais:** Modelagem de estratificação térmica em lagos e reservatórios.

### Conexão com a Teoria

> 📖 **Referência:** A equação de calor e soluções analíticas são apresentadas em Incropera et al. (2017), Cap. 5. O modelo de Côté & Konrad para condutividade térmica de solos é descrito em Côté & Konrad (2005). O esquema implícito e o algoritmo de Thomas são detalhados em Lugon Jr. (2026), Cap. 8, Seções 8.4–8.6.

---
## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 8](../notebooks/08_Transferencia_Calor_Solos.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 8.8

Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*
</div>